In [17]:
#from google.colab import drive
#drive.mount("/content/drive", force_remount=True)

In [18]:
import os
import copy
import math
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, balanced_accuracy_score
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

cuda


In [ ]:
DATA_DIR = "/content/drive/MyDrive/data/"
OUTPUT_DIR = "/content/drive/MyDrive/models/"

MODEL_NAME = "lexlms/legal-longformer-base"
MAX_CHUNKS = 8
CHUNK_LENGTH = 512
HIDDEN_SIZE = 768
SEED = 42

CONFIG = {
    "lora_r": 4,
    "lora_alpha": 8,
    "lora_dropout": 0.1,
    "hidden_dropout": 0.2,
    "head_lr_mult": 2.0,
    "lr": 5e-5,
    "batch_size": 8,
    "accum_steps": 4,
    "epochs": 15,
    "early_stop": 5,
    "warmup_ratio": 0.1
}


torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def load_data(path):
    paths = [path] if os.path.isfile(path) else glob.glob(os.path.join(path, "*.csv"))
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df = df[df["label"] != -1].copy()
    df = df[df["side_addressed"].isin([0.0, 1.0])].copy()
    df["label"] = df["label"].astype(int)
    df["side_addressed"] = df["side_addressed"].astype(int)
    return df

def build_case_df(df):
    cases = []
    for case_id, group in df.groupby("case_id"):
        pet_rows = group[group["side_addressed"] == 0].sort_values("turn_position")
        res_rows = group[group["side_addressed"] == 1].sort_values("turn_position")

        def make_chunks(rows):
            chunks = []
            for _, r in rows.iterrows():
                ctx = str(r["preceding_context"])
                utt = str(r["justice_utterance"])
                chunks.append(f"{ctx} </s> {utt}")
            return chunks

        pet_chunks = make_chunks(pet_rows) or ["[NO PETITIONER EXCHANGES]"]
        res_chunks = make_chunks(res_rows) or ["[NO RESPONDENT EXCHANGES]"]

        labels = group["label"].dropna().unique()
        if len(labels) != 1:
            raise ValueError(f"Case {case_id} has inconsistent labels: {labels}")

        cases.append({
            "case_id": case_id,
            "pet_chunks": pet_chunks,
            "res_chunks": res_chunks,
            "label": int(labels[0])
        })
    return pd.DataFrame(cases)

train = build_case_df(load_data(os.path.join(DATA_DIR, "train_df.csv")))
validation = build_case_df(load_data(os.path.join(DATA_DIR, "val_df.csv")))
test = build_case_df(load_data(os.path.join(DATA_DIR, "test_df.csv")))

print(f"Train: {len(train)}  Val: {len(validation)}  Test: {len(test)}")

pet_lens = train["pet_chunks"].apply(len)
res_lens = train["res_chunks"].apply(len)
print(f"Pet chunks — mean: {pet_lens.mean():.1f}  p95: {pet_lens.quantile(.95):.1f}  max: {pet_lens.max()}")
print(f"Res chunks — mean: {res_lens.mean():.1f}  p95: {res_lens.quantile(.95):.1f}")

Train: 2534  Val: 236  Test: 474
Pet chunks — mean: 45.2  p95: 90.0  max: 230
Res chunks — mean: 48.3  p95: 91.0


In [21]:
#Cite paper: Chunks at the start and end are better
def select_chunks(chunks, max_chunks):
    if len(chunks) <= max_chunks:
        return chunks
    n_head = max_chunks // 2
    n_tail = max_chunks - n_head
    return chunks[:n_head] + chunks[-n_tail:]

class CaseDataset(Dataset):
    def __init__(self, df, tokenizer):
        pet_ids, pet_tok_masks, pet_chunk_masks = [], [], []
        res_ids, res_tok_masks, res_chunk_masks = [], [], []

        for _, row in df.iterrows():
            p = self._encode(row["pet_chunks"], tokenizer)
            r = self._encode(row["res_chunks"], tokenizer)
            pet_ids.append(p["input_ids"])
            pet_tok_masks.append(p["attention_mask"])
            pet_chunk_masks.append(p["chunk_mask"])
            res_ids.append(r["input_ids"])
            res_tok_masks.append(r["attention_mask"])
            res_chunk_masks.append(r["chunk_mask"])

        self.pet_ids = torch.stack(pet_ids)
        self.pet_tok_masks = torch.stack(pet_tok_masks)
        self.pet_chunk_masks = torch.stack(pet_chunk_masks)
        self.res_ids = torch.stack(res_ids)
        self.res_tok_masks = torch.stack(res_tok_masks)
        self.res_chunk_masks = torch.stack(res_chunk_masks)
        self.labels = torch.tensor(df["label"].tolist(), dtype=torch.long)


    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "pet_input_ids": self.pet_ids[idx],
            "pet_tok_mask": self.pet_tok_masks[idx],
            "pet_chunk_mask": self.pet_chunk_masks[idx],
            "res_input_ids": self.res_ids[idx],
            "res_tok_mask": self.res_tok_masks[idx],
            "res_chunk_mask": self.res_chunk_masks[idx],
            "label": self.labels[idx],
        }

    @staticmethod
    def _encode(chunks, tokenizer):
        chunks = list(chunks)
        if len(chunks) == 0:
            chunks = ["[NO EXCHANGES]"]

        chunks = select_chunks(chunks, MAX_CHUNKS)
        n_real = len(chunks)

        enc = tokenizer(
            chunks,
            max_length=CHUNK_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        pad_needed = MAX_CHUNKS - n_real
        if pad_needed > 0:
            pad_id = tokenizer.pad_token_id
            if pad_id is None:
                pad_id = tokenizer.eos_token_id
            if pad_id is None:
                pad_id = 0
            pad_ids  = torch.full((pad_needed, CHUNK_LENGTH), pad_id, dtype=torch.long)
            pad_mask = torch.zeros(pad_needed, CHUNK_LENGTH, dtype=torch.long)
            enc["input_ids"] = torch.cat([enc["input_ids"], pad_ids],  dim=0)
            enc["attention_mask"] = torch.cat([enc["attention_mask"], pad_mask], dim=0)

        chunk_mask = torch.zeros(MAX_CHUNKS, dtype=torch.bool)
        chunk_mask[:n_real] = True

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "chunk_mask": chunk_mask,
        }


In [22]:
class DualEmbeddingClassifier(nn.Module):
    def __init__(self, encoder, dropout=0.2):
        super().__init__()
        self.encoder = encoder
        H = encoder.config.hidden_size

        self.token_pool_score = nn.Linear(H, 1)
        self.chunk_pool_score = nn.Linear(H, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(H * 4),
            nn.Dropout(dropout),
            nn.Linear(H * 4, H),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(H, 2),
        )

    def encode_chunks(self, input_ids, attention_mask):
        global_attn = torch.zeros_like(attention_mask)
        global_attn[:, 0] = attention_mask[:, 0]

        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attn,
        ).last_hidden_state

        scores = self.token_pool_score(hidden).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.bmm(weights.unsqueeze(1), hidden).squeeze(1)
        return torch.nan_to_num(pooled, nan=0.0)

    def encode_side(self, input_ids, tok_mask, chunk_mask):
        B, C, L = input_ids.shape

        chunk_vecs = self.encode_chunks(
            input_ids.reshape(B * C, L),
            tok_mask.reshape(B * C, L),
        ).reshape(B, C, -1)

        scores = self.chunk_pool_score(chunk_vecs).squeeze(-1)
        scores = scores.masked_fill(~chunk_mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1)
        return torch.bmm(weights.unsqueeze(1), chunk_vecs).squeeze(1)

    def forward(self,
                pet_input_ids, pet_tok_mask, pet_chunk_mask,
                res_input_ids, res_tok_mask, res_chunk_mask):
        v_pet = self.encode_side(pet_input_ids, pet_tok_mask, pet_chunk_mask)
        v_res = self.encode_side(res_input_ids, res_tok_mask, res_chunk_mask)

        features = torch.cat(
            [v_pet, v_res, torch.abs(v_pet - v_res), v_pet * v_res],
            dim=-1,
        )
        return self.classifier(features)



In [23]:
def train_epoch(model, loader, loss_fn, optimizer, scheduler, accum_steps):
    model.train()

    total_loss = 0.0
    total_examples = 0
    preds, labels = [], []

    optimizer.zero_grad()

    for step, batch in tqdm(enumerate(loader), total=len(loader), desc="Training", leave=False):
        pet_ids = batch["pet_input_ids"].to(device)
        pet_tok = batch["pet_tok_mask"].to(device)
        pet_chunk = batch["pet_chunk_mask"].to(device)

        res_ids = batch["res_input_ids"].to(device)
        res_tok = batch["res_tok_mask"].to(device)
        res_chunk = batch["res_chunk_mask"].to(device)

        lbls = batch["label"].to(device)

        logits = model(
            pet_ids, pet_tok, pet_chunk,
            res_ids, res_tok, res_chunk,
        )

        raw_loss = loss_fn(logits, lbls)
        loss = raw_loss / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or step == len(loader) - 1:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            optimizer.zero_grad()

        batch_size = lbls.size(0)
        total_loss += raw_loss.item() * batch_size
        total_examples += batch_size

        preds.extend(logits.detach().argmax(dim=-1).cpu().numpy())
        labels.extend(lbls.detach().cpu().numpy())

    avg_loss = total_loss / total_examples
    acc = accuracy_score(labels, preds)

    return avg_loss, acc

In [24]:
def evaluate(model, loader, loss_fn=None):
    model.eval()
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()
    total_loss =     0.0
    total_examples = 0
    preds, labels, probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            pet_ids = batch["pet_input_ids"].to(device)
            pet_tok = batch["pet_tok_mask"].to(device)
            pet_chunk = batch["pet_chunk_mask"].to(device)
            res_ids = batch["res_input_ids"].to(device)
            res_tok = batch["res_tok_mask"].to(device)
            res_chunk = batch["res_chunk_mask"].to(device)
            lbls = batch["label"].to(device)

            logits = model(pet_ids, pet_tok, pet_chunk, res_ids, res_tok, res_chunk)

            batch_size = lbls.size(0)
            total_loss += loss_fn(logits, lbls).item() * batch_size
            total_examples += batch_size

            preds.extend(logits.argmax(-1).cpu().numpy())
            labels.extend(lbls.cpu().numpy())
            probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())

    return total_loss / total_examples, accuracy_score(labels, preds), preds, labels, probs

In [25]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_encoder = AutoModel.from_pretrained(MODEL_NAME)

train_loader = DataLoader(CaseDataset(train,  tokenizer), batch_size=CONFIG["batch_size"],shuffle=True)
val_loader = DataLoader(CaseDataset(validation, tokenizer), batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(CaseDataset(test,  tokenizer), batch_size=CONFIG["batch_size"], shuffle=False)

encoder = get_peft_model(copy.deepcopy(base_encoder), LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["query", "value"],
    bias="none",
))


model = DualEmbeddingClassifier(encoder, dropout=CONFIG["hidden_dropout"]).to(device)

optimizer = AdamW([
    {"params": model.encoder.parameters(), "lr": CONFIG["lr"]},
    {"params": model.classifier.parameters(), "lr": CONFIG["lr"] * CONFIG["head_lr_mult"]},
    {"params": model.token_pool_score.parameters(), "lr": CONFIG["lr"] * CONFIG["head_lr_mult"]},
    {"params": model.chunk_pool_score.parameters(), "lr": CONFIG["lr"] * CONFIG["head_lr_mult"]},
], weight_decay=0.01)

num_update_steps = math.ceil(len(train_loader) / CONFIG["accum_steps"]) * CONFIG["epochs"]
warmup_steps = max(1, int(num_update_steps * CONFIG["warmup_ratio"]))

scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, num_update_steps)

loss_fn = nn.CrossEntropyLoss()
save_path  = os.path.join(OUTPUT_DIR, "global")
best_score = -float("inf")

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: lexlms/legal-longformer-base
Key                                | Status     | 
-----------------------------------+------------+-
lm_head.dense.weight               | UNEXPECTED | 
lm_head.layer_norm.weight          | UNEXPECTED | 
lm_head.dense.bias                 | UNEXPECTED | 
longformer.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                       | UNEXPECTED | 
lm_head.layer_norm.bias            | UNEXPECTED | 
pooler.dense.weight                | MISSING    | 
pooler.dense.bias                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:

print(f"\n{'Ep':>3} {'TrLoss':>8} {'TrAcc':>7} {'VaLoss':>8} {'VaAcc':>7} {'VaMF1':>7}")

for epoch in range(1, CONFIG["epochs"] + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, CONFIG["accum_steps"])
    va_loss, va_acc, va_preds, va_labels, va_probs = evaluate(model, val_loader)
    va_macro_f1 = f1_score(va_labels, va_preds, average="macro", zero_division=0)

    flag = ""
    if va_macro_f1 > best_score:
        best_score = va_macro_f1
        no_improve = 0
        os.makedirs(save_path, exist_ok=True)
        model.encoder.save_pretrained(save_path)
        torch.save(model.classifier.state_dict(), os.path.join(save_path, "classifier.pt"))
        torch.save(model.token_pool_score.state_dict(), os.path.join(save_path, "token_pool_score.pt"))
        torch.save(model.chunk_pool_score.state_dict(), os.path.join(save_path, "chunk_pool_score.pt"))
        flag = " ✓"
    else:
        no_improve += 1
        if no_improve >= CONFIG["early_stop"]:
            print(f"  Early stop at epoch {epoch}")
            break

    print(f"{epoch:>3} {tr_loss:>8.4f} {tr_acc:>7.4f} {va_loss:>8.4f} {va_acc:>7.4f} {va_macro_f1:>7.4f}{flag}")

# Reload best checkpoint
best_encoder = PeftModel.from_pretrained(
    copy.deepcopy(base_encoder),
    save_path,
)
best_model = DualEmbeddingClassifier(best_encoder, dropout=CONFIG["hidden_dropout"]).to(device)
best_model.classifier.load_state_dict(torch.load(os.path.join(save_path, "classifier.pt"), map_location=device))
best_model.token_pool_score.load_state_dict( torch.load(os.path.join(save_path, "token_pool_score.pt"), map_location=device))
best_model.chunk_pool_score.load_state_dict( torch.load(os.path.join(save_path, "chunk_pool_score.pt"), map_location=device))

# Test evaluation
_, te_acc, te_preds, te_labels, _ = evaluate(best_model, test_loader)

print(f"Test accuracy: {te_acc:.4f}")
te_labels_arr = np.array(te_labels)
baseline = max(te_labels_arr.mean(), 1 - te_labels_arr.mean())
print(f"Baseline: {baseline:.4f}")
print(classification_report(te_labels, te_preds, target_names=["respondent", "petitioner"], zero_division=0))

pd.DataFrame([{
    "test_acc": accuracy_score(te_labels, te_preds),
    "balanced_acc": balanced_accuracy_score(te_labels, te_preds),
    "macro_f1": f1_score(te_labels, te_preds, average="macro", zero_division=0),
    "baseline": baseline,
}]).to_csv(os.path.join(OUTPUT_DIR, "results.csv"), index=False)


 Ep   TrLoss   TrAcc   VaLoss   VaAcc   VaMF1


Training:   0%|          | 0/634 [00:00<?, ?it/s]

  1   0.6987  0.4980   0.6921  0.5212  0.4189 ✓


Training:   0%|          | 0/634 [00:00<?, ?it/s]

  2   0.7041  0.5059   0.6922  0.4958  0.3827


...

 13   0.6298  0.6527   0.6870  0.5720  0.5294


Training:   0%|          | 0/634 [00:00<?, ?it/s]

 14   0.6337  0.6373   0.6752  0.5890  0.5762 ✓


Training:   0%|          | 0/634 [00:00<?, ?it/s]

 15   0.6280  0.6575   0.6710  0.5932  0.5874 ✓
Test accuracy: 0.5907
Baseline: 0.5000
              precision    recall  f1-score   support

  respondent       0.57      0.78      0.66       237
  petitioner       0.65      0.40      0.49       237

    accuracy                           0.59       474
   macro avg       0.61      0.59      0.57       474
weighted avg       0.61      0.59      0.57       474

